<a href="https://colab.research.google.com/github/mutrera-2004/Semantic-Segmentation-using-Hypercolumns/blob/main/final_project_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [51]:
import torchvision
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torchvision import models
from torchvision.transforms import InterpolationMode
from torchvision.datasets import VOCSegmentation

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torch.utils.data import sampler
from torch.utils.data import DataLoader, Dataset

import os
import math
import matplotlib.pyplot as plt
import random
import numpy as np

In [52]:
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")

# Dataset and VGG

In [53]:
# Class to perform resizing and cropping of Pascal images
# Takes resize and crop dimensions -> outputs img, mask tensors resized
# We need to ensure that the img and mask correspond (same region is cropped)
class VOCTransform:
    def __init__(self, resize=256, crop=224):
        self.resize = resize
        self.crop = crop

    def __call__(self, img, mask):
        img = TF.resize(img, self.resize)
        mask = TF.resize(mask, self.resize, interpolation=InterpolationMode.NEAREST)

        i, j, h, w = T.RandomCrop.get_params(img, (self.crop, self.crop))

        img = TF.crop(img, i, j, h, w)
        mask = TF.crop(mask, i, j, h, w)

        img = TF.to_tensor(img)
        mask = TF.pil_to_tensor(mask)

        return img, mask

In [54]:
# Transform images as they load
class VOCDataset(VOCSegmentation):
    def __init__(self, root, year, image_set, transform=None, download=False):
        super().__init__(root=root, year=year, image_set=image_set, download=download)
        self.joint_transform = transform

    def __getitem__(self, index):
        img, mask = super().__getitem__(index)

        if self.joint_transform:
            img, mask = self.joint_transform(img, mask)

        return img, mask.squeeze(0).long()

In [55]:
VAL_SPLIT = 0.1
PASCAL_BATCH_SIZE = 8
NUM_WORKERS = 0

joint_transform = VOCTransform(resize=256, crop=224)

# Load data train/test splits
voc_full_train = VOCDataset(
    './data',
    year='2012',
    image_set='train',
    transform=joint_transform,
    download=True
)

voc_test = VOCDataset(
    './data',
    year='2012',
    image_set='val',
    transform=joint_transform,
    download=True
)

# Get validation split from train split
val_size = int(len(voc_full_train) * VAL_SPLIT)
train_size = len(voc_full_train) - val_size

voc_train, voc_val = random_split(voc_full_train, [train_size, val_size])

# Data loaders (for batched images)
voc_train_loader = DataLoader(
    voc_train,
    batch_size=PASCAL_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)

voc_val_loader = DataLoader(
    voc_val,
    batch_size=PASCAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

voc_test_loader = DataLoader(
    voc_test,
    batch_size=PASCAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

In [56]:
# Load pretrained VGG16
vgg_model = models.vgg16(weights=models.VGG16_Weights.DEFAULT)

# Building Hypercolumn Dataset Splits

In [57]:
def extract_feature_maps(x, model, layers = [3, 8, 15, 22, 29]):
    features = {}

    for i, layer in enumerate(model.features):
        x = layer(x)

        if i in layers:
            features[i] = x

    return features

In [58]:
vgg_model

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [59]:
def build_hypercolumns(feature_dict):
    """
    feature_dict: dict[layer_idx -> tensor[B, C, H_i, W_i]]

    Returns:
        tensor [B, Z, H, W]
        where Z = sum of all channel dimensions
    """

    # get reference spatial size (highest resolution)
    first_map = list(feature_dict.values())[0]
    B, _, H, W = first_map.shape

    upsampled_maps = []

    for fmap in feature_dict.values():
        if fmap.shape[2:] != (H, W):
            fmap = F.interpolate(
                fmap,
                size=(H, W),
                mode='bilinear',
                align_corners=False
            )

        upsampled_maps.append(fmap)

    # concatenate channels
    hypercolumns = torch.cat(upsampled_maps, dim=1)

    return hypercolumns

In [60]:
def sample_hypercolumns_per_image(hyper, label, samples_per_class=3):
    """
    hyper: torch tensor [Z, H, W] -> hypercolumns of one image
    label: torch tensor [H, W] -> class label per pixel
    samples_per_class: int, number of pixels to sample per class

    Returns:
        list of tuples: [(feature, label), ...]
            feature: [Z, 1, 1]
            label: [1, 1]
    """
    Z, H, W = hyper.shape
    sampled_list = []

    # get unique classes in this image
    classes = torch.unique(label)
    classes = classes[classes != 255] # ignore the countour from labeled images as this is not a class

    for cls in classes:
        # find pixels belonging to this class
        mask = (label == cls)  # [H, W]
        indices = mask.nonzero(as_tuple=False)  # [[i, j], ...]

        # if fewer pixels than samples_per_class, take all
        if indices.shape[0] < samples_per_class:
            selected_indices = indices
        else:
            perm = torch.randperm(indices.shape[0])
            selected_indices = indices[perm[:samples_per_class]]

        # extract hypercolumns for selected pixels
        for idx in selected_indices:
            i, j = idx
            feature = hyper[:, i, j].unsqueeze(-1).unsqueeze(-1)  # [Z, 1, 1]
            lbl = label[i, j].unsqueeze(0).unsqueeze(0)           # [1, 1]
            sampled_list.append((feature, lbl))

    return sampled_list

In [61]:
import torch
from torch.utils.data import IterableDataset

class HypercolumnPixelDataset(IterableDataset):
    def __init__(self, voc_loader, vgg_model, device='mps', samples_per_class=3):
        """
        voc_loader: DataLoader yielding (img_batch, label_batch)
        vgg_model: pretrained VGG feature extractor
        samples_per_class: #pixels per class per image
        """
        self.voc_loader = voc_loader
        self.vgg = vgg_model.eval().to(device)
        self.samples_per_class = samples_per_class
        self.device = device
        self.len = None

    def __iter__(self):
        for img_batch, lbl_batch in self.voc_loader:
            B = img_batch.shape[0]
            img_batch = img_batch.to(self.device)
            lbl_batch = lbl_batch.to(self.device)

            # 1️⃣ Forward batch through VGG
            with torch.no_grad():
                feature_maps = extract_feature_maps(img_batch, self.vgg)  # [B, ..., H, W]
                hypercolumns = build_hypercolumns(feature_maps)           # [B, Z, H, W]

            # 2️⃣ Sample pixels per image
            for i in range(B):
                hyper = hypercolumns[i]  # [Z, H, W]
                lbl = lbl_batch[i]       # [H, W]

                sampled = sample_hypercolumns_per_image(hyper, lbl, self.samples_per_class) # list[(hyper, lbl)]

                # 3️⃣ Yield each sampled pixel
                for hyp, label in sampled:
                    yield hyp.view(-1), label.view(-1) # around 8 * 22 for each VOC batch

In [62]:
HYPER_BATCH_SIZE = 64
PIXEL_SAMPLES = 3 # controls how many hypercolumn, label pairs to get per pixel class for a given sample
DEVICE = get_device()

# Hyperpixel pair dataset and loaders
train_pixel_dataset = HypercolumnPixelDataset(voc_train_loader, vgg_model, samples_per_class=PIXEL_SAMPLES, device = DEVICE)
train_pixel_loader = DataLoader(train_pixel_dataset, batch_size=HYPER_BATCH_SIZE, drop_last=True)

val_pixel_dataset = HypercolumnPixelDataset(voc_val_loader, vgg_model, samples_per_class=PIXEL_SAMPLES, device = DEVICE)
val_pixel_loader = DataLoader(val_pixel_dataset, batch_size=HYPER_BATCH_SIZE, drop_last=True)

test_pixel_dataset = HypercolumnPixelDataset(voc_test_loader, vgg_model, samples_per_class=PIXEL_SAMPLES, device = DEVICE)
test_pixel_loader = DataLoader(test_pixel_dataset, batch_size=HYPER_BATCH_SIZE, drop_last=True)

# Hypercolumn Classifier Architecture

In [63]:
class HC_Classifier(nn.Module):
    def __init__(self, in_dim=1472, num_classes=21):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(in_dim, 256, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Conv2d(256, 256, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Conv2d(256, 128, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(128, num_classes, 1)
        )

    def forward(self, x):
        x = F.normalize(x, dim=1)
        return self.net(x)

In [64]:
def val(loader, model, criterion, device):
    num_correct = 0
    num_samples = 0
    total_loss = 0
    num_batches = 0
    model.eval() # set model to evaluation mode

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device).unsqueeze(-1).unsqueeze(-1) # [B, Z, 1, 1]
            y = y.to(device).view(-1)

            out = model(x)           # [B, C, 1, 1]
            out = out.squeeze(-1).squeeze(-1)  # [B, C]
            loss = criterion(out, y) # needed to graph per epoch loss and accuracy
            preds = out.argmax(dim=1)

            num_correct += (preds == y).sum().item()
            num_samples += y.size(0)

            # for computation of per epoch avg loss and acc
            total_loss += loss.item()
            num_batches += 1

        acc = num_correct / num_samples
        print('Eval %d / %d correct (%.2f)' % (num_correct, num_samples, 100 * acc))

    avg_loss = total_loss / num_batches

    return avg_loss, acc

In [65]:
def train_hc_classifier(loader_train, loader_val, model, optimizer, device, epochs):
    model = model.to(device)
    model.train()
    criterion = torch.nn.CrossEntropyLoss()

    epoch_loss_val = []
    epoch_acc_val = []
    epoch_loss_train = []
    epoch_acc_train = []


    for e in range(epochs):
        model.train()

        # for train loss and acc validai
        train_loss_sum = 0
        train_correct = 0
        train_samples = 0
        num_batches = 0

        for t, (x, y) in enumerate(loader_train):
            x = x.to(device).unsqueeze(-1).unsqueeze(-1) # [B, Z, 1, 1] H=W=1 for training
            y = y.to(device).view(-1) # [B, ]

            out = model(x)           # [B, Z, 1, 1]
            out = out.squeeze(-1).squeeze(-1)  # [B, C]
            loss = criterion(out, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # for train loss, acc computation
            preds = out.argmax(dim=1)
            train_correct += (preds == y).sum().item()
            train_samples += y.size(0)
            train_loss_sum += loss.item()
            num_batches += 1


            if t % 64 == 0:
                print('Epoch %d, Iteration %d, loss = %.4f' % (e, t, loss.item()))

        avg_train_loss = train_loss_sum / num_batches
        train_acc = train_correct / train_samples

        epoch_loss_train.append(avg_train_loss)
        epoch_acc_train.append(train_acc)

        e_loss_val, e_acc_val = val(loader_val, model, criterion, device)

        epoch_loss_val.append(e_loss_val)
        epoch_acc_val.append(e_acc_val)

    return epoch_loss_train, epoch_acc_train, epoch_loss_val, epoch_acc_val


In [ ]:
LR = 2e-2
WEIGHT_DECAY = 1e-4
P = 0.9
EPOCHS = 50
DEVICE = get_device()

hc_classifier = HC_Classifier()
optimizer = optim.SGD(hc_classifier.parameters(), LR, P, WEIGHT_DECAY)
loss_train, acc_train, loss_val, acc_val = train_hc_classifier(train_pixel_loader, val_pixel_loader, hc_classifier, optimizer, DEVICE, EPOCHS)

Epoch 0, Iteration 0, loss = 3.2253
Epoch 0, Iteration 64, loss = 2.0228
Epoch 0, Iteration 128, loss = 2.0231
Eval 579 / 1088 correct (53.22)
Epoch 1, Iteration 0, loss = 1.8369
Epoch 1, Iteration 64, loss = 1.8919
Epoch 1, Iteration 128, loss = 1.2743
Eval 598 / 1088 correct (54.96)
Epoch 2, Iteration 0, loss = 2.0897
Epoch 2, Iteration 64, loss = 1.5753
Epoch 2, Iteration 128, loss = 1.6114
Eval 624 / 1088 correct (57.35)
Epoch 3, Iteration 0, loss = 1.4819
Epoch 3, Iteration 64, loss = 1.1292
Epoch 3, Iteration 128, loss = 1.2666
Eval 643 / 1088 correct (59.10)
Epoch 4, Iteration 0, loss = 1.5763
Epoch 4, Iteration 64, loss = 1.3391
Epoch 4, Iteration 128, loss = 1.4967
Eval 634 / 1088 correct (58.27)
Epoch 5, Iteration 0, loss = 1.1195
Epoch 5, Iteration 64, loss = 1.1927
Epoch 5, Iteration 128, loss = 1.7267
Eval 683 / 1088 correct (62.78)
Epoch 6, Iteration 0, loss = 1.2605
Epoch 6, Iteration 64, loss = 1.5122
Epoch 6, Iteration 128, loss = 1.1876
Eval 677 / 1088 correct (62.22)

In [ ]:
# torch.save(hc_classifier.state_dict())

In [ ]:
epochs = range(1, len(loss_train) + 1)

plt.figure(figsize=(12,5))

# ---- Loss plot ----
plt.subplot(1,2,1)
plt.plot(epochs, loss_train, marker='o', label="Train Loss")
plt.plot(epochs, loss_val, marker='o', label="Val Loss")
plt.title("Loss per Epoch")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

# ---- Accuracy plot ----
plt.subplot(1,2,2)
plt.plot(epochs, acc_train, marker='o', label="Train Accuracy")
plt.plot(epochs, acc_val, marker='o', label="Val Accuracy")
plt.title("Accuracy per Epoch")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

Current validation instabilities might be due to batch variation. We are normalizing the inputs to the model and that is why the loss and accuracy curves are smooth for training data.

I'll try to run a couple of inferences on the final images to observe model performance at this current state. Then we shall see if we can aim for some accuracy improvements, but it seems that the model is going to struggle still and 55% might be our roof. I might ask the professor if accuracy is normal to be capped there or if there is better.

# TEST WITH WHOLE IMAGES

In [ ]:
import torch

def compute_iou(pred, target):
    """
    pred:   Tensor [H, W] predicted labels
    target: Tensor [H, W] ground truth labels

    Returns:
        dict mapping class -> IoU
        mean IoU
    """

    classes = torch.unique(target)
    ious = {}

    for cls in classes:
        pred_c = pred == cls
        target_c = target == cls

        intersection = torch.logical_and(pred_c, target_c).sum().float()
        union = torch.logical_or(pred_c, target_c).sum().float()

        if union == 0:
            iou = torch.nan
        else:
            iou = intersection / union

        ious[int(cls)] = iou.item()

    miou = torch.tensor(list(ious.values())).mean()

    return ious, miou.item()

In [ ]:
for i, (img, lbl) in enumerate(voc_test_loader):
    if i == 10:
        break

    img = img.to(DEVICE)
    feature_maps = extract_feature_maps(img, vgg_model)
    hypercolumns = build_hypercolumns(feature_maps)
    out = hc_classifier(hypercolumns)

    test_img = img[0].cpu()
    test_lbl = lbl[0]
    test_out = out[0].cpu()
    pred_mask = torch.argmax(test_out, dim=0)

    ious, miou = compute_iou(pred_mask, test_lbl)

    fig, ax = plt.subplots(1, 3, figsize=(10,10))
    ax[0].imshow(test_img.permute(1,2,0))
    ax[0].set_title("Sample Image")
    ax[1].imshow(test_lbl)
    ax[1].set_title("Segmentation Label")
    ax[2].imshow(pred_mask)
    ax[2].set_title(f"Result \n IoU: {miou}")
    plt.show()

In [ ]:
ious, miou = compute_iou(pred_mask, test_lbl)

In [ ]:
miou * 100

# Research (Understanding of the Task)

### Dataset
Images: random images of a variety of things. **They're different sizes!**
Labels: annotated segmentations, where pixels are labeled per the object class (does pixel belong to object? -> assign some number). Outlines are 255 and background is 0

### Architecture
Pre-trained VGG architecture (a CNN with multiple layers)

### Training
For each pixel p in the original image:
1. Take feature maps from several layers.
2. Upsample them to the original resolution (bilinear interpolation).
3. Extract the vector at pixel p
4. Concatenate them.

However, we do this for a sample of pixels! Don't use all of them or else massive memory overhead; it would take forever.

(1) Instead of fine-tuning the entire system, fix the pre-trained CNN and learn a shallow classifier that acts on extracted hypercolumns. For this auxiliary classifier, you may experiment with different architectures, but general advice is to keep it below 10 layers (and maybe below 4 if training is too slow). This classifier should predict a softmax distribution over pixels, i.e. a H x W x C tensor where C is the number of classes.

(2) While at inference (test) time, your system should produce predictions for every pixel location in the image, this is unnecessary at training time.  Greater statistical efficiency during training is possible by subsampling a sparse set of locations per image at which to extract hypercolumns.  You can build a training set of hypercolumn features in an offline manner by, e.g.,
(i) For each training image, sample 3 pixels from each class.
(ii) For each pixel, save the pixel hypercolumn feature (i.e., 1x1xZ dimensional descriptor), along with the identity of that pixel as a label.
In the PASCAL VOC dataset, there are an average of 2.5 classes per image, so this process should yield around 10,000 example (feature, label) pairs.

### Loss Function

### Metrics